*italicized text*

In [ ]:
%cd drive/MyDrive/Thesis/Models/one_hot_NN

/content/drive/MyDrive/Thesis/Models/one_hot_NN


In [ ]:
!ls

Make_windowed_data.ipynb  model.py  test_RNN.ipynb  training.ipynb


In [ ]:
import numpy as np
import torch
import torch.nn as nn

from model import MLP_onehot_flat

In [ ]:
#----- training loop
def training_loop(epochs, batch_size, train_dataset, model, loss_func, opt):

  train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle=True, drop_last=True)
  iterations = len(train_dataset)//batch_size #len(dataset)//batch_size
  model.train()

  for epoch in range(epochs):
    print(f"Epoch[{epoch+1}/{epochs}]\n")
    
    cum_loss = 0
    acc_count = 0

    for x,y in train_loader:
      x = torch.nn.functional.one_hot(x.type(torch.int64), num_classes=22) #N x 31 x 22
      x = torch.flatten(x, start_dim=1, end_dim = 2)
      x = x.type(torch.float)
      
      #forward
      outputs    = model(x)
      _, preds   = torch.max(outputs, 1)
      acc_count += (preds == y).sum().item()

      loss = loss_func(outputs, y)
      cum_loss += loss.item()

      #backward
      opt.zero_grad() 
      loss.backward()

      #adam step
      opt.step()

      #scheduler step

    print(f'Loss: {(cum_loss/(iterations*batch_size)):.3f}   Accuracy: {(acc_count/(iterations*batch_size)):.3f}\n')
    # for x,y in validation_loader:


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

np_data = np.load('../../Data/np_windows.npz')
tensor_data = torch.as_tensor(np_data['np_seq'], dtype=torch.int8, device=device) # 4614740 x 31  #int8 tensor. maybe around 3-4GB. so can load entire tensor into GPU memory to speed up training
train_label = torch.as_tensor(np_data['np_labels'], dtype = torch.int64, device=device)
dataset = torch.utils.data.TensorDataset(tensor_data, train_label)
del(np_data)

In [ ]:
model   = MLP_onehot_flat().to(device)

In [ ]:

loss_func = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay = 0.01)
training_loop(epochs=20, batch_size=100, train_dataset=dataset, model=model, loss_func=loss_func, opt=optimizer)


Epoch[1/20]

Loss: 0.008   Accuracy: 0.640

Epoch[2/20]

Loss: 0.008   Accuracy: 0.653

Epoch[3/20]

Loss: 0.008   Accuracy: 0.656

Epoch[4/20]

Loss: 0.008   Accuracy: 0.657

Epoch[5/20]

Loss: 0.008   Accuracy: 0.658

Epoch[6/20]

Loss: 0.008   Accuracy: 0.658

Epoch[7/20]

Loss: 0.008   Accuracy: 0.658

Epoch[8/20]

Loss: 0.008   Accuracy: 0.658

Epoch[9/20]

Loss: 0.008   Accuracy: 0.658

Epoch[10/20]

Loss: 0.008   Accuracy: 0.658

Epoch[11/20]

Loss: 0.008   Accuracy: 0.658

Epoch[12/20]

Loss: 0.008   Accuracy: 0.658

Epoch[13/20]



KeyboardInterrupt: ignored

In [ ]:
torch.save(model.state_dict(), './saved_model')

In [ ]:
# def crossvalid(model=None,criterion=None,optimizer=None,dataset=None,k_fold=5):
    
#     train_score = pd.Series()
#     val_score = pd.Series()
    
#     total_size = len(dataset)

#     for i in range(k_fold):
#       train_loader = 
#     fraction = 1/k_fold
#     seg = int(total_size * fraction)


#     train_set = torch.utils.data.dataset.Subset(dataset,train_indices)
#     val_set = torch.utils.data.dataset.Subset(dataset,val_indices)
    
# #         print(len(train_set),len(val_set))
# #         print()
    
#     train_loader = torch.utils.data.DataLoader(train_set, batch_size=50, shuffle=True, num_workers=4)
#     val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=50, shuffle=True, num_workers=4)
#     train_acc    = train(res_model,criterion,optimizer,train_loader,epoch=1)

#     train_score.at[i] = train_acc
#     val_acc = valid(res_model,criterion,optimizer,val_loader)
#     val_score.at[i] = val_acc

#     return train_score,val_score
        

# train_score,val_score = crossvalid(res_model,criterion,optimizer,dataset=tiny_dataset)


In [ ]:
# x = torch.randint(0,  22, (5000,31))
# print(x.size())
# # y = torch.randint(0,3,(5000,))
# x = nn.functional.one_hot(x)
# print(x.size())
# x = torch.flatten(x, start_dim=1, end_dim = 2)
# print(x.size())
# # train_dataset = torch.utils.data.TensorDataset(x, y)
# # train_loader  = torch.utils.data.DataLoader(train_dataset, batch_size = 50, shuffle=True)

torch.Size([5000, 31])
torch.Size([5000, 31, 22])
torch.Size([5000, 682])
